# Demo - Tokenization: BPE and WordPiece
**Day 2 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-tokenization.ipynb)

**Goal:** Show that models see text as a sequence of integer token IDs, and
that subword tokenization splits rare words into known pieces.

We tokenize the same sentences two ways: GPT-style BPE (`tiktoken`) and
BERT-style WordPiece (`transformers`). We print the tokens, the integer IDs,
and the token count so you can see why token count, not word count, is the
real budget.

No API key needed. `tiktoken` runs offline; the BERT tokenizer downloads a
small vocab file once, then caches.

In [ ]:
# Scoped install for this demo (safe to re-run; tiktoken is preinstalled on Colab).
%pip install -q tiktoken transformers

In [ ]:
import tiktoken
from transformers import AutoTokenizer

## 1. The sentences we will tokenize
Four sentences: a plain one, one about the topic itself, a rare medical term,
and a line of code. Watch how the rare word and the code split into pieces.

In [ ]:
SENTENCES = [
    "The cat sat on the mat.",
    "Tokenization splits rare words into subwords.",
    "acetaminophen",
    "def get_weather(city): return api.fetch(city)",
]
for s in SENTENCES:
    print(s)

## 2. BPE (GPT family, via tiktoken)
`cl100k_base` is the encoding used by GPT-3.5/GPT-4 family models. It runs
fully offline, no download required.

In [ ]:
def demo_bpe():
    """GPT-style Byte Pair Encoding with tiktoken (fully offline)."""
    print("1. BPE (GPT family, via tiktoken)")
    encoder = tiktoken.get_encoding("cl100k_base")
    for text in SENTENCES:
        ids = encoder.encode(text)
        pieces = [encoder.decode([i]) for i in ids]
        print(f"\n  text   : {text!r}")
        print(f"  tokens : {pieces}")
        print(f"  ids    : {ids}")
        print(f"  count  : {len(ids)} tokens for {len(text)} characters")


demo_bpe()

## 3. WordPiece (BERT family, via transformers)
`bert-base-uncased` downloads a small vocab file once, then caches. A `##`
prefix marks a piece that attaches to the previous token.

In [ ]:
def demo_wordpiece():
    """BERT-style WordPiece with the transformers tokenizer."""
    print("2. WordPiece (BERT family, via transformers)")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    for text in SENTENCES:
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.convert_tokens_to_ids(tokens)
        print(f"\n  text   : {text!r}")
        print(f"  tokens : {tokens}")
        print(f"  ids    : {ids}")
        print(f"  count  : {len(tokens)} tokens")
    print("\n  Note: '##' marks a piece that attaches to the previous token.")


demo_wordpiece()

## Expected output
- Common words (`the`, `cat`, `sat`) stay whole tokens in both tokenizers.
- `acetaminophen` splits into several subword pieces in both tokenizers,
  since it is a rare word not in either vocabulary as a single token.
- The code line splits on punctuation and identifier boundaries.
- Token count is always higher than word count for the rare word and code
  examples, but close to word count for the plain sentence.

**Takeaway:** models count tokens, not words. Rare words cost more tokens.